# Advanced Python Single Inheritance
## 18 substantial problems, worked solutions, and executable tests

**Learning path:** subtype relationships → introspection → method lookup → constructor chains → polymorphism → abstract classes → invariants → design trade-offs → serialization → capstone testing.

**Basis and scope.** The supplied lesson introduces `Shape → Ellipse → Circle` and `Shape → Polygon → Rectangle → Square` (plus `Triangle`), `issubclass()` versus `isinstance()`, transitive inheritance, unrelated branches, and implicit inheritance from `object`. Problems 1–4 and 18 directly extend those examples. Problems 5–17 add new, clearly marked applications of single inheritance; these applications are *extensions*, not claims made by the source lesson.

**How to use:** (1) Read a problem and predict the output. (2) Solve it independently before opening the next code cell. (3) Run the worked solution, then its verification cell. (4) Change at least one input or edge case. (5) Choose **Kernel → Restart & Run All** to verify the notebook end to end. Solutions are intentionally visible and copyable; each code cell is runnable without external packages. Python 3.10+ recommended.

**Best-practice conventions:** single direct parent per custom class; explicit input validation using exceptions (not `assert` in production code); tests use `assert`/`unittest`; `super()` for cooperative overriding; meaningful class names; no network, files, third-party packages, or arbitrary deserialization. Notebook code is instructional, not a drop-in business library.

**Important distinction:** `issubclass(C, C)` is `True`, even though a class is not considered its *own direct parent*. An instance's `type(x) is C` checks exact type; `isinstance(x, C)` includes subclasses. All ordinary new-style classes ultimately inherit from `object` in Python 3.

## Setup — reproducible baseline from the supplied hierarchy
The basic empty classes match the lesson’s relationships. The shared helper makes intentionally failing examples safe to execute, so **Run All** completes normally.

In [1]:
from __future__ import annotations

import math
from decimal import Decimal


def expect_raises(exception_type, function, *args, **kwargs):
    """Return the expected exception or raise AssertionError if none occurs."""
    try:
        function(*args, **kwargs)
    except exception_type as error:
        return error
    raise AssertionError(f"Expected {exception_type.__name__}")


class Shape:
    pass


class Ellipse(Shape):
    pass


class Circle(Ellipse):
    pass


class Polygon(Shape):
    pass


class Rectangle(Polygon):
    pass


class Square(Rectangle):
    pass


class Triangle(Polygon):
    pass


SHAPE_CLASSES = (Shape, Ellipse, Circle, Polygon, Rectangle, Square, Triangle)
print('Baseline hierarchy ready:', ', '.join(cls.__name__ for cls in SHAPE_CLASSES))

Baseline hierarchy ready: Shape, Ellipse, Circle, Polygon, Rectangle, Square, Triangle


---

## Problem 01 — Build a 7 × 7 inheritance truth table  ·  Advanced foundations

**Challenge.** For every ordered pair of classes in the source hierarchy, compute both `issubclass(child, parent)` and whether `parent` is a **direct** base. Print an understandable matrix. Prove that `Square` is transitively a `Shape` but not an `Ellipse`, and that a class is its own subclass but not its own direct base.

**Hint (try before reading the solution).** Use `issubclass()` for transitive ancestry and `child.__bases__` for direct parentage. Distinguish diagonal entries.

### Worked solution

In [2]:
def subclass_matrix(classes):
    return {
        (child.__name__, parent.__name__): issubclass(child, parent)
        for child in classes
        for parent in classes
    }


def direct_parent(child, parent):
    return parent in child.__bases__


relationship_grid = subclass_matrix(SHAPE_CLASSES)
header = f"{'child \ parent':>14}" + ''.join(f'{p.__name__:>11}' for p in SHAPE_CLASSES)
print(header)
for child in SHAPE_CLASSES:
    bits = ''.join(f'{str(issubclass(child, parent)):>11}' for parent in SHAPE_CLASSES)
    print(f'{child.__name__:>14}{bits}')

child \ parent      Shape    Ellipse     Circle    Polygon  Rectangle     Square   Triangle
         Shape       True      False      False      False      False      False      False
       Ellipse       True       True      False      False      False      False      False
        Circle       True       True       True      False      False      False      False
       Polygon       True      False      False       True      False      False      False
     Rectangle       True      False      False       True       True      False      False
        Square       True      False      False       True       True       True      False
      Triangle       True      False      False       True      False      False       True


<>:14: SyntaxWarning: invalid escape sequence '\ '
<>:14: SyntaxWarning: invalid escape sequence '\ '
C:\Users\user1\AppData\Local\Temp\ipykernel_19136\19543649.py:14: SyntaxWarning: invalid escape sequence '\ '
  header = f"{'child \ parent':>14}" + ''.join(f'{p.__name__:>11}' for p in SHAPE_CLASSES)


### Verification / edge cases

In [3]:
assert len(relationship_grid) == 49
assert relationship_grid['Square', 'Shape'] is True
assert relationship_grid['Square', 'Ellipse'] is False
assert relationship_grid['Square', 'Square'] is True
assert direct_parent(Square, Rectangle)
assert not direct_parent(Square, Shape)
assert not direct_parent(Square, Square)
assert direct_parent(Shape, object)
print('PASS: 49 subclass relationships; direct vs. indirect verified.')

PASS: 49 subclass relationships; direct vs. indirect verified.


### Why this works / engineering notes

`issubclass` follows *all* ancestors and includes the class itself. `__bases__` lists only explicitly or implicitly established **immediate** base classes. For example, the direct base of `Square` is `Rectangle`, while `Shape` is an indirect ancestor. In this exercise all defined classes have a single direct parent.

---

## Problem 02 — Design safe runtime type classification  ·  Advanced foundations

**Challenge.** Given a `Square` instance and a list of possible classes, produce its compatible types in order. Compare exact type with compatible type; deliberately pass invalid operands to `issubclass` and capture the errors without interrupting the notebook. Demonstrate the tuple-of-classes feature.

**Hint (try before reading the solution).** Use `isinstance(instance, cls)`, `type(instance) is cls`, and `expect_raises(TypeError, ...)`.

### Worked solution

In [4]:
def matching_types(value, candidates):
    if not all(isinstance(candidate, type) for candidate in candidates):
        raise TypeError('All candidates must be classes')
    return [candidate.__name__ for candidate in candidates if isinstance(value, candidate)]


square_obj = Square()
print('Compatible types:', matching_types(square_obj, SHAPE_CLASSES))
print('Exact type is Rectangle:', type(square_obj) is Rectangle)
print('Compatible with Rectangle:', isinstance(square_obj, Rectangle))
print('Compatible with either Circle or Rectangle:', isinstance(square_obj, (Circle, Rectangle)))

invalid_call_error = expect_raises(TypeError, issubclass, square_obj, Shape)
print('Expected misuse:', invalid_call_error)

Compatible types: ['Shape', 'Polygon', 'Rectangle', 'Square']
Exact type is Rectangle: False
Compatible with Rectangle: True
Compatible with either Circle or Rectangle: True
Expected misuse: issubclass() arg 1 must be a class


### Verification / edge cases

In [5]:
assert matching_types(square_obj, SHAPE_CLASSES) == [
    'Shape', 'Polygon', 'Rectangle', 'Square'
]
assert type(square_obj) is Square
assert isinstance(square_obj, Shape)
assert not isinstance(square_obj, Ellipse)
assert issubclass(Square, (Ellipse, Polygon))
assert isinstance(square_obj, (Circle, Rectangle))
expect_raises(TypeError, matching_types, square_obj, [Shape, 'not a class'])
expect_raises(TypeError, issubclass, Square(), Shape)
print('PASS: exact type, compatible type, tuple alternatives, and errors.')

PASS: exact type, compatible type, tuple alternatives, and errors.


### Why this works / engineering notes

Use `isinstance` when subclasses should be accepted; exact `type(...) is ...` when the requirement truly excludes subclasses. `issubclass` requires a class in the first argument; an object instance is not a valid substitute. Both built-ins accept a tuple of candidate types in their second argument. An explicit check on candidate inputs gives a clearer API error.

---

## Problem 03 — Find inheritance distance and nearest common ancestor  ·  Advanced introspection

**Challenge.** Implement `inheritance_distance(child, ancestor)` for a single-inheritance hierarchy (0 for the same class) and `nearest_common_ancestor(a, b, include_object=False)`. Reject unrelated pairs for distance; return `None` for unrelated application classes if `object` is excluded.

**Hint (try before reading the solution).** Search `__mro__`, which lists the class followed by ancestors. For a single-inheritance chain, the first common member is the closest common ancestor.

### Worked solution

In [6]:
def inheritance_distance(child, ancestor):
    if not isinstance(child, type) or not isinstance(ancestor, type):
        raise TypeError('Both arguments must be classes')
    if not issubclass(child, ancestor):
        raise ValueError(f'{child.__name__} is not a subclass of {ancestor.__name__}')
    return child.__mro__.index(ancestor)


def nearest_common_ancestor(left, right, *, include_object=False):
    if not isinstance(left, type) or not isinstance(right, type):
        raise TypeError('Both arguments must be classes')
    return next(
        (ancestor for ancestor in left.__mro__
         if ancestor in right.__mro__ and (include_object or ancestor is not object)),
        None,
    )


class UnrelatedPerson:
    pass


print('Square → Shape distance:', inheritance_distance(Square, Shape))
print('Square + Triangle:', nearest_common_ancestor(Square, Triangle).__name__)
print('Circle + Square:', nearest_common_ancestor(Circle, Square).__name__)

Square → Shape distance: 3
Square + Triangle: Polygon
Circle + Square: Shape


### Verification / edge cases

In [7]:
assert inheritance_distance(Square, Square) == 0
assert inheritance_distance(Square, Rectangle) == 1
assert inheritance_distance(Square, Shape) == 3
assert nearest_common_ancestor(Square, Triangle) is Polygon
assert nearest_common_ancestor(Circle, Square) is Shape
assert nearest_common_ancestor(Shape, UnrelatedPerson) is None
assert nearest_common_ancestor(Shape, UnrelatedPerson, include_object=True) is object
expect_raises(ValueError, inheritance_distance, Square, Ellipse)
expect_raises(TypeError, inheritance_distance, Square(), Shape)
print('PASS: distances, common ancestors, unrelated branches, bad arguments.')

PASS: distances, common ancestors, unrelated branches, bad arguments.


### Why this works / engineering notes

Python’s method-resolution order (`__mro__`) includes indirect ancestors and ends with `object`. Here all classes use *single inheritance*, so index position also gives the number of parent edges. Treating `object` as a meaningful shared domain ancestor is a caller choice; excluding it by default avoids confusing unrelated families with a common application-specific base.

---

## Problem 04 — Discover descendant classes and leaves dynamically  ·  Advanced introspection

**Challenge.** Write a recursive descendant collector and a leaf-class detector that do not hard-code class names. Build a separate toy hierarchy and verify all descendants and terminal classes. Explain one real-world limitation of `__subclasses__()`.

**Hint (try before reading the solution).** `cls.__subclasses__()` returns *immediate, currently live* subclasses; recurse and de-duplicate.

### Worked solution

In [8]:
class TreeRoot:
    pass


class TreeBranch(TreeRoot):
    pass


class TreeLeaf(TreeBranch):
    pass


class TreeSibling(TreeRoot):
    pass


def all_descendants(root):
    if not isinstance(root, type):
        raise TypeError('root must be a class')
    found = set()
    pending = list(root.__subclasses__())
    while pending:
        child = pending.pop()
        if child not in found:
            found.add(child)
            pending.extend(child.__subclasses__())
    return found


def leaf_descendants(root):
    descendants = all_descendants(root)
    return {cls for cls in descendants if not any(
        sub in descendants for sub in cls.__subclasses__()
    )}


print('Descendants:', sorted(cls.__name__ for cls in all_descendants(TreeRoot)))
print('Leaves:', sorted(cls.__name__ for cls in leaf_descendants(TreeRoot)))

Descendants: ['TreeBranch', 'TreeLeaf', 'TreeSibling']
Leaves: ['TreeLeaf', 'TreeSibling']


### Verification / edge cases

In [9]:
assert all_descendants(TreeRoot) == {TreeBranch, TreeLeaf, TreeSibling}
assert leaf_descendants(TreeRoot) == {TreeLeaf, TreeSibling}
assert all_descendants(TreeLeaf) == set()
assert leaf_descendants(TreeLeaf) == set()
expect_raises(TypeError, all_descendants, TreeRoot())
print('PASS: recursive discovery and terminal descendants.')

PASS: recursive discovery and terminal descendants.


### Why this works / engineering notes

`__subclasses__()` only reports subclasses still alive at the moment of inspection; local, dynamically created, unloaded, or garbage-collected classes can change its results. This makes it useful for exploration, but not necessarily a dependable persistent plugin registry. A deliberate registry is implemented in Problem 15.

---

## Problem 05 — Keep counters separate across an inheritance chain  ·  State and class hooks

**Challenge.** Construct `AssetTracker → PhotoAsset → RawPhotoAsset`. Each class must count only instances whose **exact class** is that class. Demonstrate how assigning an attribute on one instance can shadow a class attribute without changing the shared counter.

**Hint (try before reading the solution).** `__init_subclass__` initializes storage for every new subclass. Update `type(self).created`, not `self.created`.

### Worked solution

In [10]:
class AssetTracker:
    created = 0

    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        cls.created = 0

    def __init__(self, label):
        if not isinstance(label, str) or not label.strip():
            raise ValueError('label must be non-empty text')
        self.label = label.strip()
        type(self).created += 1


class PhotoAsset(AssetTracker):
    pass


class RawPhotoAsset(PhotoAsset):
    pass


base_asset = AssetTracker('general')
photo_asset = PhotoAsset('JPEG')
raw_a = RawPhotoAsset('RAW 1')
raw_b = RawPhotoAsset('RAW 2')
print('Counts:', AssetTracker.created, PhotoAsset.created, RawPhotoAsset.created)

Counts: 1 1 2


### Verification / edge cases

In [11]:
assert (AssetTracker.created, PhotoAsset.created, RawPhotoAsset.created) == (1, 1, 2)
photo_asset.created = 999  # Instance shadow; not a mutation of PhotoAsset.created.
assert photo_asset.created == 999
assert PhotoAsset.created == 1
assert raw_a.created == 2  # Reads the RawPhotoAsset class attribute.
expect_raises(ValueError, PhotoAsset, '')
assert PhotoAsset.created == 1  # Failed validation did not increment the counter.
print('PASS: per-exact-class counters, shadowing, and validation.')

PASS: per-exact-class counters, shadowing, and validation.


### Why this works / engineering notes

Class attributes are inherited on **lookup**, but an assignment to `self.created` would typically create or update an instance attribute. `type(self).created += 1` instead updates the attribute on the exact runtime class; `__init_subclass__` ensures that each new subclass starts with separate class-level storage. These counts measure successful initializations, not how many instances are currently alive.

---

## Problem 06 — Trace `super()` through three overridden methods  ·  Method resolution

**Challenge.** Implement a three-level class chain. Each `describe()` method must add its own contribution while retaining the ancestor’s contribution. Print the MRO and verify call order. Use `super()` without hard-coding the parent’s name.

**Hint (try before reading the solution).** Call `super().describe()` before appending your stage; the MRO determines the next lookup target.

### Worked solution

In [12]:
class TraceBase:
    def describe(self):
        return ['base']


class TraceMiddle(TraceBase):
    def describe(self):
        result = super().describe()
        return result + ['middle']


class TraceFinal(TraceMiddle):
    def describe(self):
        return super().describe() + ['final']


trace = TraceFinal()
print('Method calls:', ' → '.join(trace.describe()))
print('MRO:', ' → '.join(cls.__name__ for cls in TraceFinal.__mro__))

Method calls: base → middle → final
MRO: TraceFinal → TraceMiddle → TraceBase → object


### Verification / edge cases

In [13]:
assert trace.describe() == ['base', 'middle', 'final']
assert TraceMiddle().describe() == ['base', 'middle']
assert TraceFinal.__mro__ == (TraceFinal, TraceMiddle, TraceBase, object)
assert issubclass(TraceFinal, TraceBase)
print('PASS: cooperative overriding and method lookup order.')

PASS: cooperative overriding and method lookup order.


### Why this works / engineering notes

`super()` does not blindly mean “invoke a function declared in the textual parent.” It performs a lookup beginning *after the current class* in the relevant MRO. In a single-inheritance chain, that usually corresponds to the next ancestor. `super()` preserves parent behavior when implementing a specialized override.

---

## Problem 07 — Build a validated three-stage constructor chain  ·  Constructors and invariants

**Challenge.** Build `Record → TimedRecord → AuditedRecord` with progressively more required state. Enforce nonblank identifiers, timezone-aware timestamps, and a nonblank audit note. Override `summary()` while preserving the parent’s result. Verify invalid inputs raise informative exceptions.

**Hint (try before reading the solution).** Each subclass should call `super().__init__(...)` before storing subclass-specific fields; treat input validation as a runtime contract.

### Worked solution

In [14]:
from datetime import datetime, timezone


class Record:
    def __init__(self, record_id):
        if not isinstance(record_id, str) or not record_id.strip():
            raise ValueError('record_id must be non-blank text')
        self.record_id = record_id.strip()

    def summary(self):
        return f'ID={self.record_id}'


class TimedRecord(Record):
    def __init__(self, record_id, occurred_at):
        super().__init__(record_id)
        if not isinstance(occurred_at, datetime) or occurred_at.utcoffset() is None:
            raise ValueError('occurred_at must be a timezone-aware datetime')
        self.occurred_at = occurred_at

    def summary(self):
        return f'{super().summary()} | at={self.occurred_at.isoformat()}'


class AuditedRecord(TimedRecord):
    def __init__(self, record_id, occurred_at, note):
        super().__init__(record_id, occurred_at)
        if not isinstance(note, str) or not note.strip():
            raise ValueError('note must be non-blank text')
        self.note = note.strip()

    def summary(self):
        return f'{super().summary()} | note={self.note}'


record = AuditedRecord('  rec-007 ', datetime(2026, 1, 2, tzinfo=timezone.utc), ' reviewed ')
print(record.summary())

ID=rec-007 | at=2026-01-02T00:00:00+00:00 | note=reviewed


### Verification / edge cases

In [15]:
assert record.record_id == 'rec-007'
assert record.note == 'reviewed'
assert 'at=2026-01-02T00:00:00+00:00' in record.summary()
assert isinstance(record, Record)
expect_raises(ValueError, Record, '  ')
expect_raises(ValueError, TimedRecord, 'ok', datetime(2026, 1, 2))
expect_raises(ValueError, AuditedRecord, 'ok', datetime.now(timezone.utc), '')
print('PASS: constructor delegation, specialization, and invalid inputs.')

PASS: constructor delegation, specialization, and invalid inputs.


### Why this works / engineering notes

Each constructor enforces its own part of the invariant and delegates the inherited part via `super()`. A subclass must not bypass parent initialization if inherited methods rely on parent fields. An aware `datetime` expresses an unambiguous instant; `.utcoffset() is None` identifies naive or otherwise timezone-unresolved input.

---

## Problem 08 — Implement a template-method reporting framework  ·  Polymorphism

**Challenge.** Write a base report whose `render()` controls the sequence of steps. A sales report supplies its own header and rows; a monthly sales report adds a footer. Keep orchestration in the base so that subclass methods are called polymorphically without branching on types.

**Hint (try before reading the solution).** A template method calls overridable hooks on `self`; `MonthlySalesReport` can build on `SalesReport.footer()` using `super()`.

### Worked solution

In [16]:
class ReportTemplate:
    def heading(self):
        return 'Report'

    def rows(self):
        raise NotImplementedError('Subclasses must supply report rows')

    def footer(self):
        return 'End of report'

    def render(self):
        parts = [self.heading(), *self.rows(), self.footer()]
        return '\n'.join(parts)


class SalesReport(ReportTemplate):
    def __init__(self, sales):
        if not isinstance(sales, (list, tuple)) or any(
            isinstance(amount, bool) or not isinstance(amount, int) or amount < 0
            for amount in sales
        ):
            raise ValueError('sales must be a sequence of nonnegative integers')
        self.sales = tuple(sales)

    def heading(self):
        return 'Sales report'

    def rows(self):
        return [f'Item {idx}: {amount}' for idx, amount in enumerate(self.sales, 1)]


class MonthlySalesReport(SalesReport):
    def __init__(self, sales, month):
        super().__init__(sales)
        if not isinstance(month, str) or not month.strip():
            raise ValueError('month is required')
        self.month = month.strip()

    def heading(self):
        return f'{super().heading()} — {self.month}'

    def footer(self):
        return f'Total: {sum(self.sales)}\n{super().footer()}'


monthly = MonthlySalesReport([25, 40, 35], 'January')
print(monthly.render())

Sales report — January
Item 1: 25
Item 2: 40
Item 3: 35
Total: 100
End of report


### Verification / edge cases

In [17]:
lines = monthly.render().splitlines()
assert lines[0] == 'Sales report — January'
assert lines[1:4] == ['Item 1: 25', 'Item 2: 40', 'Item 3: 35']
assert lines[-2:] == ['Total: 100', 'End of report']
assert isinstance(monthly, ReportTemplate)
expect_raises(NotImplementedError, ReportTemplate().render)
expect_raises(ValueError, SalesReport, [10, -1])
print('PASS: dynamic method dispatch and preserved rendering algorithm.')

PASS: dynamic method dispatch and preserved rendering algorithm.


### Why this works / engineering notes

`render()` belongs to the base, but calls `self.heading()`, `self.rows()`, and `self.footer()`. Python dispatches each call to the most-specific implementation. The base defines the *stable algorithm*; specialized classes provide variable behavior. Avoid checking `type(self)` in the base to decide which subclass-specific code should run.

---

## Problem 09 — Enforce an abstract contract without multiple inheritance  ·  Abstraction

**Challenge.** Build `Notifier → EmailNotifier → TransactionalEmailNotifier`. `Notifier` must not be instantiable until `send()` is implemented. Fake all sending: do not contact real networks. Show abstract enforcement, inherited implementation, and a specialized override.

**Hint (try before reading the solution).** Use `ABC` and `@abstractmethod` in the root and implement `send` in the first concrete child.

### Worked solution

In [18]:
from abc import ABC, abstractmethod


class Notifier(ABC):
    @abstractmethod
    def send(self, recipient, body):
        """Return a description of a simulated send."""


class EmailNotifier(Notifier):
    def send(self, recipient, body):
        if not isinstance(recipient, str) or '@' not in recipient:
            raise ValueError('recipient must resemble an email address')
        if not isinstance(body, str):
            raise TypeError('body must be text')
        return f'SIMULATED EMAIL to {recipient}: {body}'


class TransactionalEmailNotifier(EmailNotifier):
    def send(self, recipient, body):
        if not isinstance(body, str):
            raise TypeError('body must be text')
        return super().send(recipient, f'[transactional] {body}')


notifier = TransactionalEmailNotifier()
print(notifier.send('example@example.com', 'Receipt created'))

SIMULATED EMAIL to example@example.com: [transactional] Receipt created


### Verification / edge cases

In [19]:
expect_raises(TypeError, Notifier)
assert isinstance(EmailNotifier(), Notifier)
assert issubclass(TransactionalEmailNotifier, Notifier)
assert '[transactional] Receipt created' in notifier.send('a@b.com', 'Receipt created')
expect_raises(ValueError, notifier.send, 'bad-address', 'hello')
expect_raises(TypeError, notifier.send, 'a@b.com', None)
assert len(TransactionalEmailNotifier.__bases__) == 1
print('PASS: abstract contract, concrete implementation, safe simulation.')

PASS: abstract contract, concrete implementation, safe simulation.


### Why this works / engineering notes

`ABC` provides a reusable abstract base; adding `@abstractmethod` prevents incomplete implementations from being instantiated. Inheriting an implementation satisfies the abstract contract for descendants. Real email addresses require fuller validation and a dedicated delivery service; the intentionally simple validator here is only to demonstrate inheritance.

---

## Problem 10 — Protect a property invariant and inherit a factory method  ·  Properties and class methods

**Challenge.** Create a temperature class that always represents a finite Celsius reading at or above absolute zero. Add a property setter, a Fahrenheit class factory, and a labeled subclass that inherits the factory. Ensure factory construction still works with the subclass’s extra required argument.

**Hint (try before reading the solution).** Define the factory as `@classmethod`; call `cls(celsius, **kwargs)` rather than naming `Temperature` explicitly.

### Worked solution

In [20]:
class Temperature:
    ABSOLUTE_ZERO_C = -273.15

    def __init__(self, celsius):
        self.celsius = celsius

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError('temperature must be numeric (excluding bool)')
        if not math.isfinite(value) or value < self.ABSOLUTE_ZERO_C:
            raise ValueError('temperature must be finite and not below absolute zero')
        self._celsius = float(value)

    @classmethod
    def from_fahrenheit(cls, fahrenheit, **kwargs):
        if isinstance(fahrenheit, bool) or not isinstance(fahrenheit, (int, float)):
            raise TypeError('fahrenheit must be numeric')
        return cls((fahrenheit - 32) * 5 / 9, **kwargs)


class LabeledTemperature(Temperature):
    def __init__(self, celsius, label):
        super().__init__(celsius)
        if not isinstance(label, str) or not label.strip():
            raise ValueError('label is required')
        self.label = label.strip()


freezing = LabeledTemperature.from_fahrenheit(32, label='freezing point')
print(type(freezing).__name__, freezing.celsius, freezing.label)

LabeledTemperature 0.0 freezing point


### Verification / edge cases

In [21]:
assert isinstance(freezing, Temperature)
assert type(freezing) is LabeledTemperature
assert freezing.celsius == 0.0
assert math.isclose(Temperature.from_fahrenheit(212).celsius, 100.0)
expect_raises(ValueError, Temperature, float('nan'))
expect_raises(ValueError, Temperature, -274)
expect_raises(TypeError, Temperature, True)
expect_raises(ValueError, setattr, freezing, 'celsius', float('inf'))
assert freezing.celsius == 0.0  # Setter failed without corrupting stored state.
expect_raises(TypeError, LabeledTemperature.from_fahrenheit, 32)
print('PASS: validated property, dynamic class factory, subtype preservation.')

PASS: validated property, dynamic class factory, subtype preservation.


### Why this works / engineering notes

A property enforces validation on both construction and later writes. A classmethod is bound to the *runtime class on which it is invoked*, so the inherited factory uses `LabeledTemperature` when called through that subclass. Forwarding documented `**kwargs` allows the caller to supply subclass-specific constructor inputs, but each class still owns its parameter contract.

---

## Problem 11 — Validate frozen inherited dataclasses  ·  Modern Python

**Challenge.** Build a three-stage dataclass chain for invoice lines. Base lines validate integer quantity and finite nonnegative `Decimal` price; discounted lines validate a rate in `[0,1]`; member lines require a membership ID. Implement a polymorphic `total` property, preserve parent `__post_init__`, and prove instances cannot be reassigned.

**Hint (try before reading the solution).** Every descendant must call `super().__post_init__()`; use `@dataclass(frozen=True)` consistently throughout the chain.

### Worked solution

In [22]:
from dataclasses import FrozenInstanceError, dataclass


@dataclass(frozen=True)
class InvoiceLine:
    sku: str
    quantity: int
    unit_price: Decimal

    def __post_init__(self):
        if not isinstance(self.sku, str) or not self.sku.strip():
            raise ValueError('sku is required')
        if isinstance(self.quantity, bool) or not isinstance(self.quantity, int) or self.quantity <= 0:
            raise ValueError('quantity must be a positive integer')
        if (not isinstance(self.unit_price, Decimal)
                or not self.unit_price.is_finite() or self.unit_price < 0):
            raise ValueError('unit_price must be a finite, nonnegative Decimal')

    @property
    def total(self):
        return self.unit_price * self.quantity


@dataclass(frozen=True)
class DiscountedInvoiceLine(InvoiceLine):
    discount_rate: Decimal

    def __post_init__(self):
        super().__post_init__()
        rate = self.discount_rate
        if not isinstance(rate, Decimal) or not rate.is_finite() or not (0 <= rate <= 1):
            raise ValueError('discount_rate must be a finite Decimal in [0, 1]')

    @property
    def total(self):
        return super().total * (Decimal('1') - self.discount_rate)


@dataclass(frozen=True)
class MemberInvoiceLine(DiscountedInvoiceLine):
    member_id: str

    def __post_init__(self):
        super().__post_init__()
        if not isinstance(self.member_id, str) or not self.member_id.strip():
            raise ValueError('member_id is required')


line = MemberInvoiceLine('SKU-A', 3, Decimal('19.99'), Decimal('0.10'), 'M-42')
print('Line total:', line.total)

Line total: 53.9730


### Verification / edge cases

In [23]:
assert line.total == Decimal('53.9730')
assert isinstance(line, InvoiceLine)
expect_raises(FrozenInstanceError, setattr, line, 'quantity', 9)
expect_raises(ValueError, InvoiceLine, 'SKU', True, Decimal('1'))
expect_raises(ValueError, InvoiceLine, 'SKU', 2, Decimal('NaN'))
expect_raises(ValueError, DiscountedInvoiceLine, 'SKU', 2, Decimal('10'), Decimal('1.1'))
expect_raises(ValueError, MemberInvoiceLine, 'SKU', 2, Decimal('10'), Decimal('0.1'), '')
assert line.quantity == 3
print('PASS: inherited validation, decimal arithmetic, frozen assignment.')

PASS: inherited validation, decimal arithmetic, frozen assignment.


### Why this works / engineering notes

Dataclass-generated constructors initialize inherited fields, but the final class’s `__post_init__` must explicitly delegate so earlier validation still runs. `frozen=True` prevents ordinary attribute reassignment (it is not a deep security boundary for nested mutable objects). `Decimal` avoids binary floating-point surprises in monetary examples; actual monetary systems additionally require a deliberate rounding/currency policy.

---

## Problem 12 — Design a precise exception hierarchy  ·  Exception polymorphism

**Challenge.** Create `ParseProblem → FieldProblem → MissingFieldProblem`, all descending ultimately from `Exception` through one direct parent. Implement a parser for `name` and `age` and catch the most specific errors before the general ancestor. Demonstrate ancestor catch compatibility and exception chaining.

**Hint (try before reading the solution).** Exception handlers are checked from top to bottom: place `MissingFieldProblem` before `FieldProblem` before `ParseProblem`.

### Worked solution

In [24]:
class ParseProblem(Exception):
    pass


class FieldProblem(ParseProblem):
    pass


class MissingFieldProblem(FieldProblem):
    pass


def parse_person_row(row):
    if not isinstance(row, dict):
        raise ParseProblem('row must be a dictionary')
    for field in ('name', 'age'):
        if field not in row:
            raise MissingFieldProblem(f'missing field: {field}')
    if not isinstance(row['name'], str) or not row['name'].strip():
        raise FieldProblem('name must be nonblank text')
    try:
        age = int(row['age'])
    except (ValueError, TypeError) as error:
        raise FieldProblem('age must be parseable as an integer') from error
    if isinstance(row['age'], bool) or age < 0:
        raise FieldProblem('age must be nonnegative and not bool')
    return {'name': row['name'].strip(), 'age': age}


def categorize_parse(row):
    try:
        return ('ok', parse_person_row(row))
    except MissingFieldProblem as error:
        return ('missing', str(error))
    except FieldProblem as error:
        return ('invalid field', str(error))
    except ParseProblem as error:
        return ('invalid row', str(error))


for candidate in ({'name': 'Ada', 'age': '36'}, {'name': 'Ada'},
                  {'name': '', 'age': 36}, ['wrong structure']):
    print(categorize_parse(candidate))

('ok', {'name': 'Ada', 'age': 36})
('missing', 'missing field: age')
('invalid field', 'name must be nonblank text')
('invalid row', 'row must be a dictionary')


### Verification / edge cases

In [25]:
assert categorize_parse({'name': 'Ada', 'age': '36'})[0] == 'ok'
assert categorize_parse({'name': 'Ada'})[0] == 'missing'
assert categorize_parse({'name': '', 'age': 36})[0] == 'invalid field'
assert categorize_parse([])[0] == 'invalid row'
assert issubclass(MissingFieldProblem, ParseProblem)
assert isinstance(expect_raises(MissingFieldProblem, parse_person_row, {'age': 1}), ParseProblem)
chained = expect_raises(FieldProblem, parse_person_row, {'name': 'A', 'age': 'not an int'})
assert isinstance(chained.__cause__, ValueError)
print('PASS: specific-before-general handling, subtype exceptions, chaining.')

PASS: specific-before-general handling, subtype exceptions, chaining.


### Why this works / engineering notes

Inheritance makes a specialized exception catchable through any of its parent exception classes. Handler order matters because `except ParseProblem` would also match `MissingFieldProblem`. `raise ... from error` preserves the original conversion failure as `__cause__`, aiding debugging without leaking a raw low-level exception to callers.

---

## Problem 13 — Find and repair a substitutability trap  ·  API design / LSP

**Challenge.** A mutable square inherits a mutable rectangle and changes *both* dimensions when only width is resized. Show that it violates a rectangle-client expectation, then redesign rectangles and squares as independent subclasses of a small common shape interface. The new API should expose only behavior that all shapes can honor.

**Hint (try before reading the solution).** Do **not** assert that the intentionally flawed model succeeds. Capture the broken postcondition as a Boolean, then use a sibling-class redesign.

### Worked solution

In [26]:
class MutableRectangle:
    def __init__(self, width, height):
        self.width, self.height = width, height

    def resize_width(self, new_width):
        self.width = new_width


class MutableSquare(MutableRectangle):
    def __init__(self, side):
        super().__init__(side, side)

    def resize_width(self, new_width):
        super().resize_width(new_width)
        self.height = new_width  # Square invariant conflicts with rectangle-client expectation.


def rectangle_client_preserves_height(rectangle):
    old_height = rectangle.height
    rectangle.resize_width(10)
    return rectangle.height == old_height


broken_postcondition = rectangle_client_preserves_height(MutableSquare(4))
print('Does the mutable-square substitution meet the client contract?', broken_postcondition)


class AreaShape(ABC):
    @abstractmethod
    def area(self):
        pass


class SafeRectangle(AreaShape):
    def __init__(self, width, height):
        if width <= 0 or height <= 0:
            raise ValueError('dimensions must be positive')
        self.width, self.height = width, height

    def area(self):
        return self.width * self.height


class SafeSquare(AreaShape):
    def __init__(self, side):
        if side <= 0:
            raise ValueError('side must be positive')
        self.side = side

    def area(self):
        return self.side ** 2


def combined_area(shapes):
    return sum(shape.area() for shape in shapes)


print('Safe combined area:', combined_area([SafeRectangle(2, 3), SafeSquare(4)]))

Does the mutable-square substitution meet the client contract? False
Safe combined area: 22


### Verification / edge cases

In [27]:
assert broken_postcondition is False  # Expected: the counterexample really breaks the contract.
assert combined_area([SafeRectangle(2, 3), SafeSquare(4)]) == 22
assert issubclass(SafeRectangle, AreaShape)
assert issubclass(SafeSquare, AreaShape)
assert not issubclass(SafeSquare, SafeRectangle)
expect_raises(ValueError, SafeSquare, 0)
print('PASS: defect reproduced, safer shared interface verified.')

PASS: defect reproduced, safer shared interface verified.


### Why this works / engineering notes

An “is-a” mathematical statement does not automatically justify every *mutable API* inheritance relationship. If a rectangle client expects width changes not to alter height, a square subclass maintaining equal sides cannot preserve that contract. Shared `area()` behavior is sound in a common abstract base with `Rectangle` and `Square` as siblings. This is a design illustration, not a claim that every rectangle/square model must use this particular API.

---

## Problem 14 — Use `__slots__` across a single-inheritance chain  ·  Memory model

**Challenge.** Create three classes with one new slotted field per level. Demonstrate inherited attributes still work and unexpected attributes are rejected. Inspect the base chain and explain why `__slots__` is not a strong immutability guarantee.

**Hint (try before reading the solution).** Each descendant declares `__slots__` for its **new** fields, not the fields already declared by ancestors.

### Worked solution

In [28]:
class SlotRecord:
    __slots__ = ('key',)

    def __init__(self, key):
        self.key = key


class SlotAudit(SlotRecord):
    __slots__ = ('actor',)

    def __init__(self, key, actor):
        super().__init__(key)
        self.actor = actor


class SlotSignedAudit(SlotAudit):
    __slots__ = ('signature',)

    def __init__(self, key, actor, signature):
        super().__init__(key, actor)
        self.signature = signature


audit = SlotSignedAudit('r-01', 'analyst', 'demo-signature')
print('All inherited slots:', audit.key, audit.actor, audit.signature)
print('MRO:', [cls.__name__ for cls in SlotSignedAudit.__mro__])

All inherited slots: r-01 analyst demo-signature
MRO: ['SlotSignedAudit', 'SlotAudit', 'SlotRecord', 'object']


### Verification / edge cases

In [29]:
assert audit.key == 'r-01' and audit.actor == 'analyst'
assert not hasattr(audit, '__dict__')
expect_raises(AttributeError, setattr, audit, 'unexpected', 123)
assert SlotSignedAudit.__slots__ == ('signature',)
assert issubclass(SlotSignedAudit, SlotRecord)
audit.signature = 'changed'  # Slots permit reassignment unless extra measures are taken.
assert audit.signature == 'changed'
print('PASS: inherited slots, restricted attribute names, mutability caveat.')

PASS: inherited slots, restricted attribute names, mutability caveat.


### Why this works / engineering notes

Slots restrict ordinary per-instance attribute dictionaries and can reduce overhead for many small objects. A slotted subclass should list only its own fields; a subclass without `__slots__` could reintroduce an instance `__dict__`. Slots alone do **not** make values frozen, do not validate field contents, and do not provide a security boundary.

---

## Problem 15 — Build a registry with `__init_subclass__`  ·  Extensible architecture

**Challenge.** Implement an exporter registry in which subclasses declare a unique `format_name`. Register each subclass at definition time, reject missing and duplicate keys, and provide three concrete classes in a *single-inheritance* tree: `Exporter → CsvExporter → VerboseCsvExporter` plus `JsonExporter → Exporter`. Use proper CSV escaping.

**Hint (try before reading the solution).** Inspect `cls.__dict__` so every subclass must explicitly declare its own key; a dictionary on the base stores named registrations.

### Worked solution

In [30]:
import csv
import io
import json


class Exporter:
    registry = {}

    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        key = cls.__dict__.get('format_name')
        if not isinstance(key, str) or not key.strip():
            raise ValueError(f'{cls.__name__} must define a nonempty format_name')
        if key in Exporter.registry:
            raise ValueError(f'duplicate exporter key: {key}')
        Exporter.registry[key] = cls

    def render(self, records):
        raise NotImplementedError


class CsvExporter(Exporter):
    format_name = 'csv'

    def render(self, records):
        output = io.StringIO(newline='')
        writer = csv.writer(output)
        writer.writerow(['name', 'value'])
        for record in records:
            writer.writerow([record['name'], record['value']])
        return output.getvalue()


class VerboseCsvExporter(CsvExporter):
    format_name = 'csv-verbose'

    def render(self, records):
        return '# exported with csv-verbose\n' + super().render(records)


class JsonExporter(Exporter):
    format_name = 'json'

    def render(self, records):
        return json.dumps(records, sort_keys=True)


def export_with(key, records):
    if key not in Exporter.registry:
        raise ValueError(f'unknown format: {key}')
    return Exporter.registry[key]().render(records)


sample_rows = [{'name': 'Ada, Lovelace', 'value': 7}]
print('Registered:', sorted(Exporter.registry))
print(export_with('csv', sample_rows).strip())

Registered: ['csv', 'csv-verbose', 'json']
name,value
"Ada, Lovelace",7


### Verification / edge cases

In [31]:
assert set(Exporter.registry) == {'csv', 'csv-verbose', 'json'}
assert '"Ada, Lovelace"' in export_with('csv', sample_rows)
assert export_with('csv-verbose', sample_rows).startswith('# exported with csv-verbose')
assert json.loads(export_with('json', sample_rows)) == sample_rows
expect_raises(ValueError, export_with, 'xml', sample_rows)

def make_duplicate():
    class DuplicateCsvExporter(Exporter):
        format_name = 'csv'
    return DuplicateCsvExporter


def make_missing():
    class MissingKeyExporter(Exporter):
        pass
    return MissingKeyExporter

expect_raises(ValueError, make_duplicate)
expect_raises(ValueError, make_missing)
assert set(Exporter.registry) == {'csv', 'csv-verbose', 'json'}
print('PASS: registry creation, safe CSV serialization, duplicate rejection.')

PASS: registry creation, safe CSV serialization, duplicate rejection.


### Why this works / engineering notes

`__init_subclass__` runs when a subclass is created, providing an explicit place for registration and constraints. `cls.__dict__.get(...)` demands a new key even on a descendant, preventing accidental reuse of an inherited key. Unlike `__subclasses__()` discovery, the registry retains explicit references until removed; in production, also plan for module import order, deregistration, and concurrency. CSV writers avoid unsafe manual comma/quote concatenation.

---

## Problem 16 — Round-trip a hierarchy through safe JSON data  ·  Serialization design

**Challenge.** Build `Message → UserMessage → PriorityMessage` and a sibling `SystemMessage → Message`. Serialize objects to JSON-compatible dictionaries with a type tag and deserialize using an explicit allowlist. Reject unknown tags and invalid priorities; verify that a grandchild remains a grandchild after round-trip.

**Hint (try before reading the solution).** Use a fixed mapping of allowed tags to constructors. Never dynamically import a class or `eval` a string from JSON.

### Worked solution

In [32]:
class Message:
    kind = 'message'

    def __init__(self, text):
        if not isinstance(text, str) or not text.strip():
            raise ValueError('text must be nonblank')
        self.text = text.strip()

    def to_record(self):
        return {'kind': self.kind, 'text': self.text}


class UserMessage(Message):
    kind = 'user'

    def __init__(self, text, username):
        super().__init__(text)
        if not isinstance(username, str) or not username.strip():
            raise ValueError('username must be nonblank')
        self.username = username.strip()

    def to_record(self):
        return {**super().to_record(), 'username': self.username}


class PriorityMessage(UserMessage):
    kind = 'priority'

    def __init__(self, text, username, priority):
        super().__init__(text, username)
        if isinstance(priority, bool) or not isinstance(priority, int) or not 1 <= priority <= 5:
            raise ValueError('priority must be an integer from 1 to 5')
        self.priority = priority

    def to_record(self):
        return {**super().to_record(), 'priority': self.priority}


class SystemMessage(Message):
    kind = 'system'


MESSAGE_FACTORIES = {
    'message': lambda data: Message(data['text']),
    'user': lambda data: UserMessage(data['text'], data['username']),
    'priority': lambda data: PriorityMessage(data['text'], data['username'], data['priority']),
    'system': lambda data: SystemMessage(data['text']),
}


def message_from_record(data):
    if not isinstance(data, dict) or data.get('kind') not in MESSAGE_FACTORIES:
        raise ValueError('unknown or missing message kind')
    try:
        return MESSAGE_FACTORIES[data['kind']](data)
    except KeyError as error:
        raise ValueError(f'missing message field: {error.args[0]}') from error


def message_round_trip(message):
    return message_from_record(json.loads(json.dumps(message.to_record())))


original = PriorityMessage('ship update', 'ada', 4)
restored = message_round_trip(original)
print('JSON:', json.dumps(original.to_record(), sort_keys=True))
print('Restored exact class:', type(restored).__name__)

JSON: {"kind": "priority", "priority": 4, "text": "ship update", "username": "ada"}
Restored exact class: PriorityMessage


### Verification / edge cases

In [33]:
for value in (Message('hello'), UserMessage('hello', 'ada'),
              PriorityMessage('hello', 'ada', 2), SystemMessage('notice')):
    restored_value = message_round_trip(value)
    assert type(restored_value) is type(value)
    assert restored_value.to_record() == value.to_record()
assert type(restored) is PriorityMessage
assert isinstance(restored, Message)
expect_raises(ValueError, message_from_record, {'kind': '__import__', 'text': 'x'})
expect_raises(ValueError, message_from_record, {'kind': 'priority', 'text': 'x'})
expect_raises(ValueError, PriorityMessage, 'ok', 'user', True)
print('PASS: all four types round-trip; unknown and missing data rejected.')

PASS: all four types round-trip; unknown and missing data rejected.


### Why this works / engineering notes

The explicit `kind` discriminator solves the problem of preserving the **exact** subclass rather than merely reconstructing the root type. Most-specific instances can also satisfy parent `isinstance` checks, so blindly serializing by the first matching parent can lose fields. The allowlist prevents arbitrary class lookup from untrusted data; production systems should also specify schema versions, allowable extra fields, and size limits.

---

## Problem 17 — Capstone: polymorphic product quotes and shopping cart  ·  Integrated application

**Challenge.** Design `Product → DigitalProduct → SubscriptionProduct` and `Product → PhysicalProduct`. Store monetary amounts as validated `Decimal`s, reject invalid integer quantities, implement `quote()` polymorphically, and write a cart that handles all three concrete product kinds **without** inspecting their types. Include invalid-input tests.

**Hint (try before reading the solution).** Make the cart depend only on the inherited `Product.quote()` interface; use `super().quote()` for refinements. Use *composition* for the shopping cart rather than inheriting from a product.

### Worked solution

In [34]:
class Product:
    def __init__(self, sku, base_price):
        if not isinstance(sku, str) or not sku.strip():
            raise ValueError('sku is required')
        if (not isinstance(base_price, Decimal) or not base_price.is_finite()
                or base_price < 0):
            raise ValueError('base_price must be a finite, nonnegative Decimal')
        self.sku = sku.strip()
        self._base_price = base_price

    def quote(self):
        """Return the per-line-item quote in the same currency as base_price."""
        return self._base_price


class DigitalProduct(Product):
    def __init__(self, sku, base_price, download_mb):
        super().__init__(sku, base_price)
        if isinstance(download_mb, bool) or not isinstance(download_mb, int) or download_mb <= 0:
            raise ValueError('download_mb must be a positive integer')
        self.download_mb = download_mb


class SubscriptionProduct(DigitalProduct):
    def __init__(self, sku, monthly_price, download_mb, months):
        super().__init__(sku, monthly_price, download_mb)
        if isinstance(months, bool) or not isinstance(months, int) or months <= 0:
            raise ValueError('months must be a positive integer')
        self.months = months

    def quote(self):
        return super().quote() * self.months


class PhysicalProduct(Product):
    def __init__(self, sku, base_price, shipping_fee):
        super().__init__(sku, base_price)
        if (not isinstance(shipping_fee, Decimal) or not shipping_fee.is_finite()
                or shipping_fee < 0):
            raise ValueError('shipping_fee must be a finite, nonnegative Decimal')
        self.shipping_fee = shipping_fee

    def quote(self):
        return super().quote() + self.shipping_fee


class ShoppingCart:
    def __init__(self):
        self._items = []

    def add(self, product, quantity=1):
        if not isinstance(product, Product):
            raise TypeError('product must be a Product')
        if isinstance(quantity, bool) or not isinstance(quantity, int) or quantity <= 0:
            raise ValueError('quantity must be a positive integer')
        self._items.append((product, quantity))

    def total(self):
        return sum((product.quote() * quantity for product, quantity in self._items),
                   start=Decimal('0'))


cart = ShoppingCart()
cart.add(DigitalProduct('EBOOK', Decimal('12.50'), 25), 2)
cart.add(SubscriptionProduct('PRO', Decimal('7.00'), 100, 3))
cart.add(PhysicalProduct('MUG', Decimal('8.00'), Decimal('2.50')), 2)
print('Cart total:', cart.total())

Cart total: 67.00


### Verification / edge cases

In [35]:
assert cart.total() == Decimal('67.00')  # 25 + 21 + (10.5 * 2)
assert ShoppingCart().total() == Decimal('0')
assert isinstance(SubscriptionProduct('S', Decimal('1'), 1, 2), Product)
assert issubclass(SubscriptionProduct, DigitalProduct)
assert not issubclass(PhysicalProduct, DigitalProduct)
expect_raises(ValueError, SubscriptionProduct, 'S', Decimal('1'), 1, 0)
expect_raises(ValueError, PhysicalProduct, 'P', Decimal('1'), Decimal('-0.01'))
expect_raises(ValueError, Product, 'X', Decimal('NaN'))
expect_raises(ValueError, cart.add, DigitalProduct('X', Decimal('1'), 1), True)
expect_raises(TypeError, cart.add, 'not a product')
print('PASS: polymorphic pricing, cart total, exact Decimal values, invalid inputs.')

PASS: polymorphic pricing, cart total, exact Decimal values, invalid inputs.


### Why this works / engineering notes

The cart sends the same `quote()` message to every `Product`; Python chooses the right override at runtime. `SubscriptionProduct` inherits through an indirect parent, illustrating transitivity with meaningful behavior. The cart’s **has-a** relationship avoids incorrect “cart is a product” inheritance. Prices here are illustrative and intentionally omit tax, currency conversion, refunds, rounding, and mutable-price policies; real financial code needs explicit domain requirements.

---

## Problem 18 — Write a systematic inheritance contract test suite  ·  Mastery / testing

**Challenge.** Prove relationships across *all* 49 ordered pairs from the original 7-class hierarchy. For every class, create an instance and confirm `isinstance(instance, ancestor)` matches the ancestor’s membership in `cls.__mro__`. Also test reflexivity, branch separation, and `object` inheritance. Run as a real `unittest` suite inside Jupyter without discovering unrelated tests.

**Hint (try before reading the solution).** Use `unittest.TestCase.subTest` for useful failure context and pass an explicit suite to `TextTestRunner`.

### Worked solution

In [36]:
import sys
import unittest


class SingleInheritanceContractTests(unittest.TestCase):
    def test_full_class_and_instance_matrix(self):
        for child in SHAPE_CLASSES:
            instance = child()
            for candidate in SHAPE_CLASSES:
                with self.subTest(child=child.__name__, candidate=candidate.__name__):
                    expected = candidate in child.__mro__
                    self.assertEqual(issubclass(child, candidate), expected)
                    self.assertEqual(isinstance(instance, candidate), expected)

    def test_reflexive_and_direct_relationships(self):
        for cls in SHAPE_CLASSES:
            with self.subTest(cls=cls.__name__):
                self.assertTrue(issubclass(cls, cls))
                self.assertNotIn(cls, cls.__bases__)
                self.assertEqual(len(cls.__bases__), 1)

    def test_separate_branches_and_object(self):
        self.assertFalse(issubclass(Square, Ellipse))
        self.assertFalse(isinstance(Square(), Circle))
        self.assertTrue(issubclass(Circle, object))
        self.assertTrue(isinstance(Triangle(), object))
        self.assertEqual(Square.__mro__[-1], object)


suite = unittest.defaultTestLoader.loadTestsFromTestCase(SingleInheritanceContractTests)
result = unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(suite)

test_full_class_and_instance_matrix (__main__.SingleInheritanceContractTests.test_full_class_and_instance_matrix) ... ok
test_reflexive_and_direct_relationships (__main__.SingleInheritanceContractTests.test_reflexive_and_direct_relationships) ... ok
test_separate_branches_and_object (__main__.SingleInheritanceContractTests.test_separate_branches_and_object) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.009s

OK


### Verification / edge cases

In [37]:
assert result.wasSuccessful(), 'An inheritance contract test failed.'
assert result.testsRun == 3
assert len(SHAPE_CLASSES) ** 2 == 49
print('PASS: complete 49-pair class/instance matrix and 3 test methods.')

PASS: complete 49-pair class/instance matrix and 3 test methods.


### Why this works / engineering notes

The MRO provides an independent representation of ancestor membership for this ordinary single-inheritance hierarchy. Iterating over every pair is a compact way to catch false assumptions about transitivity, reflexivity, or unrelated branches. `unittest.subTest` reports which class pair failed without stopping the entire matrix. The suite is explicitly scoped to its own class for repeatable notebook execution.

---
## Concept checkpoint and next-step challenges

**Reconstruct from memory (no supplied code):**

1. Write `is_direct_subclass(child, parent)` and explain why it differs from `issubclass` for reflexive/indirect pairs.
2. Modify `nearest_common_ancestor` to return the entire common-ancestor list, ordered closest-first from the first argument.
3. Add a custom rule to the product capstone without using `if isinstance(product, SubscriptionProduct)` inside the cart.
4. Replace a mutable subclass relation with sibling classes when a child cannot honor a parent’s documented postconditions.
5. Extend `SingleInheritanceContractTests` to check the capstone’s class hierarchy.
6. Explain precisely why `object` appears in the MRO even when it was not written explicitly in a class declaration.

**Useful self-review:** Re-run the notebook from a clean kernel and check that every `PASS` appears and `unittest` reports success. The `expect_raises` helper checks expected failures without letting them interrupt execution. An `assert` is suitable for these exercises and tests; use real exceptions for validation in application methods. This notebook deliberately uses one inheritance parent for each user-defined class, while sometimes importing standard-library roots such as `ABC` or `Exception`.

**Provenance:** The seven empty shape classes and baseline conceptual claims come from the user-provided single-inheritance lesson; the richer examples, exercises, explanations, tests, and domain models are newly constructed extensions.